In [28]:
from pyspark.sql.types import * 
from pyspark.sql.functions import * 
from pyspark.sql import SparkSession

In [29]:
# create the spark session 
spark = SparkSession.builder.master('spark://172.22.0.2:7077').appName('Data Frames Demo').getOrCreate()

In [30]:
blogs_schema = StructType([
    StructField('Campaigns', ArrayType(StringType(), True), True),
    StructField('First', StringType(), True), 
    StructField('Hits', IntegerType(), True), 
    StructField('Id', IntegerType(), True), 
    StructField('Last', StringType(), True), 
    StructField('Published', StringType(), True), 
    StructField('Url', StringType(), True)]
)

In [49]:
json_path = '/opt/spark-data/blogs.json'
blogs = spark.read.schema(blogs_schema).option('multiLine', False).json(json_path)

In [50]:
blogs.show(truncate=False)

+----------------------------+---------+-----+---+-------+---------+-----------------+
|Campaigns                   |First    |Hits |Id |Last   |Published|Url              |
+----------------------------+---------+-----+---+-------+---------+-----------------+
|[web, twitter, FB, LinkedIn]|Matei    |40578|5  |Zaharia|5/14/2014|https://tinyurl.5|
|[web, twitter, FB, LinkedIn]|Denny    |7659 |3  |Lee    |6/7/2019 |https://tinyurl.3|
|[twitter, LinkedIn]         |Brooke   |8908 |2  |Wenig  |5/5/2018 |https://tinyurl.2|
|[twitter, LinkedIn]         |Reynold  |25568|6  |Xin    |3/2/2015 |https://tinyurl.6|
|[twitter, LinkedIn]         |Jules    |4535 |1  |Damji  |1/4/2016 |https://tinyurl.1|
|[twitter, FB]               |Tathagata|10568|4  |Das    |5/12/2018|https://tinyurl.4|
+----------------------------+---------+-----+---+-------+---------+-----------------+



In [51]:
print(blogs.printSchema())

root
 |-- Campaigns: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- First: string (nullable = true)
 |-- Hits: integer (nullable = true)
 |-- Id: integer (nullable = true)
 |-- Last: string (nullable = true)
 |-- Published: string (nullable = true)
 |-- Url: string (nullable = true)

None


In [52]:
print(blogs.schema)

# produce the following output 
'''
StructType([
    StructField('Campaigns', ArrayType(StringType(), True), True),
    StructField('First', StringType(), True), 
    StructField('Hits', LongType(), True), 
    StructField('Id', LongType(), True), 
    StructField('Last', StringType(), True), 
    StructField('Published', StringType(), True), 
    StructField('Url', StringType(), True)]
)
'''


StructType([StructField('Campaigns', ArrayType(StringType(), True), True), StructField('First', StringType(), True), StructField('Hits', IntegerType(), True), StructField('Id', IntegerType(), True), StructField('Last', StringType(), True), StructField('Published', StringType(), True), StructField('Url', StringType(), True)])


"\nStructType([\n    StructField('Campaigns', ArrayType(StringType(), True), True),\n    StructField('First', StringType(), True), \n    StructField('Hits', LongType(), True), \n    StructField('Id', LongType(), True), \n    StructField('Last', StringType(), True), \n    StructField('Published', StringType(), True), \n    StructField('Url', StringType(), True)]\n)\n"

In [54]:
blogs.select('first', 'last', 'Hits').show()

+---------+-------+-----+
|    first|   last| Hits|
+---------+-------+-----+
|    Matei|Zaharia|40578|
|    Denny|    Lee| 7659|
|   Brooke|  Wenig| 8908|
|  Reynold|    Xin|25568|
|    Jules|  Damji| 4535|
|Tathagata|    Das|10568|
+---------+-------+-----+



In [55]:
blogs.filter('hits > 10000').show()

+--------------------+---------+-----+---+-------+---------+-----------------+
|           Campaigns|    First| Hits| Id|   Last|Published|              Url|
+--------------------+---------+-----+---+-------+---------+-----------------+
|[web, twitter, FB...|    Matei|40578|  5|Zaharia|5/14/2014|https://tinyurl.5|
| [twitter, LinkedIn]|  Reynold|25568|  6|    Xin| 3/2/2015|https://tinyurl.6|
|       [twitter, FB]|Tathagata|10568|  4|    Das|5/12/2018|https://tinyurl.4|
+--------------------+---------+-----+---+-------+---------+-----------------+



In [56]:
blogs.where(col('hits')>10000).show()

+--------------------+---------+-----+---+-------+---------+-----------------+
|           Campaigns|    First| Hits| Id|   Last|Published|              Url|
+--------------------+---------+-----+---+-------+---------+-----------------+
|[web, twitter, FB...|    Matei|40578|  5|Zaharia|5/14/2014|https://tinyurl.5|
| [twitter, LinkedIn]|  Reynold|25568|  6|    Xin| 3/2/2015|https://tinyurl.6|
|       [twitter, FB]|Tathagata|10568|  4|    Das|5/12/2018|https://tinyurl.4|
+--------------------+---------+-----+---+-------+---------+-----------------+



In [57]:
blogs.where('hits>10000').show()

+--------------------+---------+-----+---+-------+---------+-----------------+
|           Campaigns|    First| Hits| Id|   Last|Published|              Url|
+--------------------+---------+-----+---+-------+---------+-----------------+
|[web, twitter, FB...|    Matei|40578|  5|Zaharia|5/14/2014|https://tinyurl.5|
| [twitter, LinkedIn]|  Reynold|25568|  6|    Xin| 3/2/2015|https://tinyurl.6|
|       [twitter, FB]|Tathagata|10568|  4|    Das|5/12/2018|https://tinyurl.4|
+--------------------+---------+-----+---+-------+---------+-----------------+



In [58]:
blogs.withColumn('BigHiter', col('hits')>10000).show()

+--------------------+---------+-----+---+-------+---------+-----------------+--------+
|           Campaigns|    First| Hits| Id|   Last|Published|              Url|BigHiter|
+--------------------+---------+-----+---+-------+---------+-----------------+--------+
|[web, twitter, FB...|    Matei|40578|  5|Zaharia|5/14/2014|https://tinyurl.5|    true|
|[web, twitter, FB...|    Denny| 7659|  3|    Lee| 6/7/2019|https://tinyurl.3|   false|
| [twitter, LinkedIn]|   Brooke| 8908|  2|  Wenig| 5/5/2018|https://tinyurl.2|   false|
| [twitter, LinkedIn]|  Reynold|25568|  6|    Xin| 3/2/2015|https://tinyurl.6|    true|
| [twitter, LinkedIn]|    Jules| 4535|  1|  Damji| 1/4/2016|https://tinyurl.1|   false|
|       [twitter, FB]|Tathagata|10568|  4|    Das|5/12/2018|https://tinyurl.4|    true|
+--------------------+---------+-----+---+-------+---------+-----------------+--------+



In [64]:
blogs.select(
    avg('hits').alias('avg_hits'),
    min('hits').alias('Min_hits'),
    max('hits').alias('Max_hits'),
    sum('hits').alias('Sum_hits'),
    count('*').alias('count_bloggers'),
).show()

+------------------+--------+--------+--------+--------------+
|          avg_hits|Min_hits|Max_hits|Sum_hits|count_bloggers|
+------------------+--------+--------+--------+--------------+
|16302.666666666666|    4535|   40578|   97816|             6|
+------------------+--------+--------+--------+--------------+



In [84]:
blogs_exploded = blogs.select(explode(blogs['Campaigns']).alias('Campaign'))\
.groupBy('Campaign')\
.count()\
.orderBy(col('count').desc())\
.show()


+--------+-----+
|Campaign|count|
+--------+-----+
| twitter|    6|
|LinkedIn|    5|
|      FB|    3|
|     web|    2|
+--------+-----+



In [90]:
blogs.withColumn('FullName', concat(col('First'), lit(' '), col('Last'))).select('FullName', 'Hits').show() 

+-------------+-----+
|     FullName| Hits|
+-------------+-----+
|Matei Zaharia|40578|
|    Denny Lee| 7659|
| Brooke Wenig| 8908|
|  Reynold Xin|25568|
|  Jules Damji| 4535|
|Tathagata Das|10568|
+-------------+-----+



In [93]:
blogs.select('First', col('Campaigns')[0].alias('First_Comapaign')).show()

+---------+---------------+
|    First|First_Comapaign|
+---------+---------------+
|    Matei|            web|
|    Denny|            web|
|   Brooke|        twitter|
|  Reynold|        twitter|
|    Jules|        twitter|
|Tathagata|        twitter|
+---------+---------------+



In [94]:
blogs.select('first').distinct().count()

6

In [96]:
blogs.filter( (col('hits') > 10000) & (col('published').contains('2014')) ).show()

+--------------------+-----+-----+---+-------+---------+-----------------+
|           Campaigns|First| Hits| Id|   Last|Published|              Url|
+--------------------+-----+-----+---+-------+---------+-----------------+
|[web, twitter, FB...|Matei|40578|  5|Zaharia|5/14/2014|https://tinyurl.5|
+--------------------+-----+-----+---+-------+---------+-----------------+



In [97]:
spark.stop()